# Planted-bug benchmark (piece B)

Breaks each valid task's reference patch in one small way (7 operators, `src/g4swe/faults.py`) and verifies every mutant with the official harness. A mutant that still resolves is a broken fix the tests cannot catch.

Code: https://github.com/TechNaom/gemma4-agentic-swe

In [ ]:
import glob, os, subprocess, sys
from pathlib import Path

LOG = '/kaggle/working/notebook.log'

def sh(cmd, check=True):
    # Tee output to notebook.log: Kaggle's API log is often empty.
    with open(LOG, 'a') as log:
        log.write(f'$ {cmd}\n')
        p = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        log.write(p.stdout + f'[exit {p.returncode}]\n')
    print('$', cmd, '\n', p.stdout[-3000:], flush=True)
    if check and p.returncode:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')

def find_one(pattern):
    hits = glob.glob(pattern, recursive=True)
    assert hits, f'not found: {pattern}'
    return Path(hits[0])

sh('git clone -q --depth 1 https://github.com/TechNaom/gemma4-agentic-swe /kaggle/working/repo')
os.chdir('/kaggle/working/repo')
WHEELHOUSE = find_one('/kaggle/input/**/gemma-4-developer-agent-wheelhouse').resolve()
DATA_DIR = find_one('/kaggle/input/**/gemma-4-developer-agent/tasks.jsonl').parent
print('wheelhouse:', WHEELHOUSE, '| data:', DATA_DIR)
os.environ['PYTHONPATH'] = 'src'
sh('pip install -q "google-adk==1.36.1" litellm networkx python-dotenv cachetools docker')
harness = [str(w) for w in WHEELHOUSE.glob('*.whl') if w.name.split('-')[0] in ('swegemma', 'adk_submission', 'adk_eval_core')]
sh('pip install -q --no-deps ' + ' '.join(harness))

In [ ]:
sh(f'python -m g4swe.faults --tasks {DATA_DIR}/tasks.jsonl --task-file experiments/valid_tasks.txt --out faults/patches')
sh(f'python -m g4swe.run_eval --mode patch --patch-dir faults/patches --data-dir {DATA_DIR} --task-file experiments/valid_tasks.txt', check=False)
sh('cp faults/manifest.jsonl results/')

In [ ]:
import json
sh('tar czf /kaggle/working/results.tgz results')
for f in sorted(glob.glob('results/*/runs.jsonl')):
    rows = [json.loads(line) for line in open(f)]
    print(f, sum(r['resolved'] for r in rows), '/', len(rows))
    for r in rows:
        print('   ', r)